# Validación de etiquetas contra la señal

**Para qué sirve:** comprobar que las etiquetas que salen de Unity coinciden con lo que hace el músculo, y decidir
si los márgenes de segmentación (`margins` en `settings.yaml`) son correctos.

**Cómo usarlo:** cambia `SESSION_DIR` y `TRIAL`, y ejecuta todo. No escribe nada en `data/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

# Las rutas de config/ y data/ son relativas a la raíz del repositorio: nos movemos a ella.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Raíz del repo:", ROOT)

In [ ]:
SESSION_DIR = "data/raw/subject_01/session_01"   # <- cambia esto
TRIAL = 0               # trial a inspeccionar en detalle (0 .. n_trials-1)
PRE_S, POST_S = 3.0, 7.0   # segundos a mostrar antes de hold_start y después del fin del gesto
ACTIVE_THRESHOLD = 3.0  # veces el nivel de reposo para fijar el instante de inicio de la activación
MIN_MEAN_ACT = 2.0      # activación media mínima (veces el reposo) para considerar que el gesto "se hizo"
UNIT_SCALE = 1e6        # V -> µV

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display

from src.config import load_yaml
from src.io import load_session
from src.protocol import label_session, protocol_params
from src.segmentation import preprocess_emg
from src.analysis.signal_diagnostics import envelope_matrix, COLORS, ORDER

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

cfg = load_yaml("experiment/settings.yaml")
CH, margins = cfg["channel_names"], cfg["margins"]

s = load_session(SESSION_DIR)
fs = s.report["emg"]["fs_est_hz"]
intervals, labels = label_session(s)
params = protocol_params(s.events)
filt = preprocess_emg(s.emg, fs, cfg["preprocessing"], CH)

env = envelope_matrix(filt[CH].to_numpy(), fs)       # envolvente RMS de 100 ms
act = env / np.median(env, axis=0)                   # en veces el nivel típico de cada canal
t = filt["t_ns"].to_numpy()

esperados, encontrados = params.get("reps", 0) * params.get("gestures", 0), int((intervals.phase == "Gesto").sum())
print(f"{s.subject} / {s.session}")
print(f"Gestos esperados según Unity: {esperados} | encontrados: {encontrados}",
      "OK" if esperados == encontrados else "<- REVISAR")
print("Márgenes en uso:", margins)

## Intervalos del protocolo
Las duraciones deben coincidir con las del protocolo (cuenta regresiva y gesto ≈ 3 s, reposo ≈ 5 s). El reposo
inicial (`rep = -1`) suele ser más largo.

In [ ]:
iv = intervals.assign(dur_s=(intervals.t_end_ns - intervals.t_start_ns) / 1e9)
display(iv.groupby("label")["dur_s"].agg(["count", "mean", "std", "min", "max"]).round(2))
display(pd.crosstab(iv["rep"], iv["label"]).rename_axis("repetición"))

## Un trial en detalle
- **Gris**: señal filtrada. **Negro**: envolvente RMS (100 ms).
- **Colores de fondo**: fase según Unity. `t = 0` es `hold_start`.
- **Barra verde** al pie de cada gráfica: la parte del intervalo que se **conserva** después de aplicar los márgenes.

Lo esperable: la actividad sube poco después del inicio del gesto (verde del gesto empieza cuando ya subió) y el
tramo verde de Reposo empieza cuando la envolvente ya volvió al nivel basal.

In [ ]:
# Vista completa: EMG + etiquetas de Unity sobre toda la sesión

t = (s.emg["t_ns"].to_numpy() - s.emg["t_ns"].iloc[0]) / 1e9
fig, axes = plt.subplots(len(CH), 1, figsize=(18, 2.2 * len(CH)), sharex=True)
for ax, c in zip(axes, CH):
    ax.plot(t, filt[c].to_numpy() * UNIT_SCALE, lw=0.5, color="tab:blue")
    for _, v in intervals.iterrows():
        a = (v.t_start_ns - s.emg["t_ns"].iloc[0]) / 1e9
        b = (v.t_end_ns - s.emg["t_ns"].iloc[0]) / 1e9
        ax.axvspan(a, b, color=COLORS.get(v.label, "gray"), alpha=0.2, lw=0)
    ax.set_ylabel(f"{c} (µV)")
axes[-1].set_xlabel("s desde el inicio del EMG")
fig.suptitle("EMG completo + sincronización con etiquetas de Unity")
fig.tight_layout()


In [ ]:
g = intervals[(intervals.phase == "Gesto") & (intervals.trial == TRIAL)]
if g.empty:
    raise ValueError(f"No hay un Gesto con trial={TRIAL}. Trials disponibles: "
                     f"{sorted(intervals.loc[intervals.phase == 'Gesto', 'trial'])}")
g = g.iloc[0]

t_ns = filt["t_ns"].to_numpy()
ref = g.t_start_ns
lo, hi = ref - PRE_S * 1e9, g.t_end_ns + POST_S * 1e9
mk = (t_ns >= lo) & (t_ns <= hi)
if not np.any(mk):
    raise ValueError(f"No hay muestras del trial {TRIAL} en el rango [{lo / 1e9:.2f}s, {hi / 1e9:.2f}s]")

x = (t_ns[mk] - ref) / 1e9
X = filt[CH].to_numpy()[mk] * UNIT_SCALE
E = env[mk] * UNIT_SCALE
near = intervals[(intervals.t_end_ns > lo) & (intervals.t_start_ns < hi)]

fig, axes = plt.subplots(len(CH), 1, figsize=(14, 2.1 * len(CH)), sharex=True)
for j, (ax, c) in enumerate(zip(axes, CH)):
    for _, v in near.iterrows():
        a, b = (v.t_start_ns - ref) / 1e9, (v.t_end_ns - ref) / 1e9
        ax.axvspan(a, b, color=COLORS[v.label], alpha=0.5, lw=0)
        mg = margins.get(v.phase) or {}
        onset_s = mg.get("onset_s")
        offset_s = mg.get("offset_s")
        if onset_s is not None and offset_s is not None:
            ax.axvspan(a + float(onset_s), b - float(offset_s), ymin=0, ymax=0.05, color="green")
    ax.plot(x, X[:, j], lw=0.4, color="gray")
    ax.plot(x, E[:, j], lw=1.4, color="black")
    ax.set_ylabel(f"{c} (µV)")
axes[-1].set_xlabel("s desde hold_start")
fig.legend(handles=[Patch(color=COLORS[l], label=l) for l in ORDER], loc="upper center", ncol=6)
fig.suptitle(f"Trial {TRIAL}: {g.label} (rep {g.rep})", y=1.0, x=0.1)
fig.tight_layout(rect=(0, 0, 1, 0.95))


## Todos los gestos: ¿hay actividad y cuándo empieza?
Para cada gesto se calcula, en veces el nivel de reposo de cada canal:
- la **activación media** por canal dentro del tramo que se conserva (después del margen de inicio),
- el **pico**, y
- la **latencia**: segundos entre `hold_start` y el primer instante en que algún canal supera `ACTIVE_THRESHOLD`
  (un valor negativo significa que la persona empezó antes de la señal).

Un gesto con `activo = False` no produjo actividad medible en ningún canal: o no se hizo, o el montaje no lo
captura. Eso **no es un error de etiquetado**, es información sobre qué tan detectable es esa expresión.

In [ ]:
rows = []
for _, g in intervals[intervals.phase == "Gesto"].iterrows():
    onset_s = (margins.get("Gesto") or {}).get("onset_s")
    if onset_s is None:
        onset_s = 0.0
    a = g.t_start_ns + float(onset_s) * 1e9
    t_ns = filt["t_ns"].to_numpy()
    i0, i1 = np.searchsorted(t_ns, [a, g.t_end_ns])
    j0, j1 = np.searchsorted(t_ns, [g.t_start_ns - 0.5e9, g.t_end_ns])
    if i1 <= i0 or j1 <= j0:
        rows.append({"trial": g.trial, "rep": g.rep, "gesto": g.label,
                     **{c: np.nan for c in CH},
                     "pico": np.nan,
                     "latencia_s": np.nan})
        continue
    over = np.flatnonzero(act[j0:j1].max(axis=1) > ACTIVE_THRESHOLD)
    rows.append({"trial": g.trial, "rep": g.rep, "gesto": g.label,
                 **dict(zip(CH, act[i0:i1].mean(axis=0))),
                 "pico": act[i0:i1].max(),
                 "latencia_s": (t_ns[j0 + over[0]] - g.t_start_ns) / 1e9 if len(over) else np.nan})
trials = pd.DataFrame(rows)
if not trials.empty:
    trials["activo"] = trials[CH].max(axis=1) >= MIN_MEAN_ACT
    display(trials.round(2).style.background_gradient(subset=CH, cmap="viridis"))
else:
    print("No hay gestos con datos para resumir.")


In [ ]:
resumen = trials.groupby("gesto").agg(
    trials=("trial", "size"),
    sin_actividad=("activo", lambda v: int((~v).sum())),
    latencia_mediana_s=("latencia_s", "median"),
    pico_mediano=("pico", "median")).round(2)
display(resumen)
display(trials.groupby("gesto")[CH].mean().round(1).style.background_gradient(cmap="viridis", axis=None)
        .set_caption("Activación media por gesto y canal (× nivel de reposo)"))

trials.boxplot(column="latencia_s", by="gesto", figsize=(7, 3.6))
plt.suptitle(""); plt.title("Latencia de activación tras hold_start"); plt.ylabel("s")

## Cómo interpretar y qué hacer

| Si ves... | Significa... | Acción |
|---|---|---|
| Gestos esperados ≠ encontrados | Falta o sobra algún evento en `Unity_events.csv` | Revisar el CSV de Unity |
| La envolvente sube *antes* del fondo de color del gesto | Desfase entre Unity y EMG | Revisar sincronización de relojes |
| La barra verde del Gesto incluye la subida | `margins.Gesto.onset_s` es corto | Aumentarlo |
| La barra verde del Reposo incluye la bajada | `margins.Reposo.onset_s` es corto | Aumentarlo (contamina la clase Reposo) |
| Un gesto con `sin_actividad` alto | La expresión casi no activa estos canales | Esperar que el modelo la confunda con Reposo |
| Dos gestos con el mismo patrón de canales | Poca separabilidad entre ellos | Esperar confusión entre ambos |

Tras cambiar `margins`, vuelve a correr `python -m scripts.prepare_data` para regenerar el dataset.